Chunking Strategy 1 - Fixed Chunk 


In [5]:
import re
import numpy as np

def fixed_chunk(text, size=300, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + size
        chunks.append(' '.join(words[start:end]))
        start += size - overlap
    return chunks

Strategy 2 - Sentence based 

In [6]:
def sentence_chunk(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    i=0
    while i<len(sentences):
        group = sentences[i:i+5]
        chunks.append(' '.join(group))
        i += 4
    return chunks

Strategy 3 - Sliding Window Chunk

In [7]:
def sliding_window_chunk(text, window=400, step=100):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + window
        chunks.append(' '.join(words[start:end]))
        start += step
    return chunks

Statistics

In [8]:
f = open("Leadership and Management importance in Organisations.txt", "r", encoding="utf-8")
text = f.read()


def chunk_stats(chunks, name):
    lengths = [len(c.split()) for c in chunks]
    print(f"{name}:")
    print(f"  Chunks: {len(chunks)}")
    print(f"  Mean length: {np.mean(lengths):.0f} words")
    print(f"  Min: {min(lengths)}, Max: {max(lengths)}\n")

fixed_chunks    = fixed_chunk(text)
sentence_chunks = sentence_chunk(text)
sliding_chunks  = sliding_window_chunk(text)

chunk_stats(fixed_chunks,    "Fixed-size")
chunk_stats(sentence_chunks, "Sentence-based")
chunk_stats(sliding_chunks,  "Sliding window")

Fixed-size:
  Chunks: 17
  Mean length: 282 words
  Min: 26, Max: 300

Sentence-based:
  Chunks: 61
  Mean length: 82 words
  Min: 8, Max: 119

Sliding window:
  Chunks: 41
  Mean length: 378 words
  Min: 26, Max: 400



Retrieval and Hit rate

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

#Embedding all three chunks 
fixed_embeddings = model.encode(fixed_chunks)
sentence_embeddings = model.encode(sentence_chunks)
sliding_embeddings = model.encode(sliding_chunks)

#Retrieving
def cosine_similarity(a,b):
 dot_prod = np.dot(a,b)
 norm_a = np.linalg.norm(a)
 norm_b = np.linalg.norm(b)
 return dot_prod/(norm_a*norm_b)

def retrieve_from(query, chunks_list, embeddings, top_k=3):
    query_emb = model.encode(query)
    scores = []
    for i, emb in enumerate(embeddings):
        score = cosine_similarity(query_emb, emb)
        scores.append((score, i))
    scores.sort(reverse=True)
    return [(chunks_list[idx], score) for score, idx in scores[:top_k]]

# 5 question-answer pairs from the document
qa_pairs = [
    ("What are the five elements of emotional intelligence?", 
     "self-awareness, self-regulation, motivation, empathy, and social skills"),
    ("Who developed the concept of emotional intelligence elements?", 
     "Goleman"),
    ("What leadership styles did Kurt Lewin identify?", 
     "autocratic, democratic, laissez-faire"),
    ("How much is spent on training managers in the UK every year?", 
     "50 billion"),
    ("What company started a training facility at an affiliate college?", 
     "Bentley")
]

# Hit rate for each strategy
def hit_rate(qa_pairs, chunks_list, embeddings):
    hits = 0
    for question, answer in qa_pairs:
        results = retrieve_from(question, chunks_list, embeddings, top_k=3)
        retrieved_text = ' '.join([chunk.lower() for chunk, _ in results])
        if answer.lower() in retrieved_text:
            hits += 1
    return hits

fixed_hits   = hit_rate(qa_pairs, fixed_chunks, fixed_embeddings)
sentence_hits = hit_rate(qa_pairs, sentence_chunks, sentence_embeddings)
sliding_hits  = hit_rate(qa_pairs, sliding_chunks, sliding_embeddings)

print(f"{'Strategy':<20} {'Chunks':<10} {'Mean Len':<12} {'Hit Rate'}")
print("-" * 55)
print(f"{'Fixed-size':<20} {len(fixed_chunks):<10} {'282 wds':<12} {fixed_hits}/5")
print(f"{'Sentence-based':<20} {len(sentence_chunks):<10} {'82 wds':<12} {sentence_hits}/5")
print(f"{'Sliding window':<20} {len(sliding_chunks):<10} {'378 wds':<12} {sliding_hits}/5")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Strategy             Chunks     Mean Len     Hit Rate
-------------------------------------------------------
Fixed-size           17         282 wds      4/5
Sentence-based       61         82 wds       4/5
Sliding window       41         378 wds      4/5
